In [7]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
from evedesign.system import System, Protein
from evedesign.models.boltzfold import BoltzFoldTransformer
from evedesign.utils import ensure_sequence

In [2]:
seq = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"
s = System([Protein(rep=seq, id='EcCM', first_index=2)])
inst = s.rep_to_instance()

In [3]:
m = BoltzFoldTransformer(device='cpu', use_msa_server=True, diffusion_samples=1)
m.build(s)

In [4]:
output_structures = m.transform([inst])

Processing 1 inputs with 1 threads.


  0%|          | 0/1 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_ae4ulaxc/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


100%|██████████| 1/1 [00:04<00:00,  4.15s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-27 12:25:32.418 | INFO     | evedesign.models.boltzfold:_load_model:208 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-27 12:30:38.480 | INFO     | evedesign.models.boltzfold:transform:371 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_ae4ulaxc/predictions
2026-04-27 12:30:38.483 | INFO     | evedesign.models.boltzfold:transform:372 - Files written (6):
2026-04-27 12:30:38.485 | INFO     | evedesign.models.boltzfold:transform:375 -   instance_0/confidence_instance_0_model_0.json (444 bytes)
2026-04-27 12:30:38.486 | INFO     | evedesign.models.boltzfold:transform:375 -   instance_0/instance_0_model_0.cif (67

In [ ]:
result = output_structures[0]
print(f"Score (boltz2 confidence score): {result.score}")

Score (ptm): 0.9246770739555359
Confidence (pLDDT): 0.9413085579872131


In [8]:
print("Confidence scores:")
for key, value in result.metadata.items():
    print(f"  {key}: {value}")

# Structure from EntityInstance.models
ei = result[0]
structures = ensure_sequence(ei.models["model_0"])

if ei.models:
    chain_id = structures[0].chains()[0]
    structure = structures[0]              
    print(f"\nChain: {chain_id}")
    print(f"Atom count: {len(structure.atom_array)}")
    print(f"Residue range: {structure.atom_array.res_id.min()} - {structure.atom_array.res_id.max()}")
    print(structure.atom_df().head(5))

Confidence scores:
  boltz_confidence: {'confidence_score': 0.9246770739555359, 'ptm': 0.8581513166427612, 'iptm': 0.0, 'ligand_iptm': 0.0, 'protein_iptm': 0.0, 'complex_plddt': 0.9413085579872131, 'complex_iplddt': 0.9413085579872131, 'complex_pde': 0.35231518745422363, 'complex_ipde': 0.0, 'chains_ptm': {'0': 0.8581513166427612}, 'pair_chains_iptm': {'0': {'0': 0.8581513166427612}}}

Chain: A
Atom count: 762
Residue range: 2 - 95
  chain_id  res_id ins_code res_name  hetero atom_name element  atom_id  \
0        A       2               THR   False         N       N        1   
1        A       2               THR   False        CA       C        2   
2        A       2               THR   False         C       C        3   
3        A       2               THR   False         O       O        4   
4        A       2               THR   False        CB       C        5   

   b_factor  occupancy  charge          x        y        z  
0    62.895        1.0       0 -44.109310  1.64779 

In [9]:
! pip install py3Dmol -q

In [11]:
import io
import py3Dmol

ei = result[0]
if ei.models:
    chain_id = list(ei.models.keys())[0]
    structure = ei.models[chain_id]

    buf = io.StringIO()
    structure.to_file(buf, format="cif")
    cif_content = buf.getvalue()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(cif_content, "cif")
    view.setStyle({
        "cartoon": {
            "colorscheme": {
                "prop": "b",
                "gradient": "roygb",
                "min": 50,
                "max": 90
            }
        }
    })
    view.zoomTo()
    view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.